In [1]:
# ============================================================
# KAGGLE ONE-CELL SCRIPT
# Version-A Howard-style ice benchmark
# ICL ONLY: mixed-K per episode + right-padding to 29 + padding mask
#
# PURPOSE
# -------
# Re-run ONLY the ICL model for Sec. 3.4.1 using the same dataset,
# architecture, optimizer, training budget, and fixed K=5 test protocol
# as the previous reference-benchmark ICL run, while changing the
# variable-K batching implementation to match the unified paper method:
#
#   - each episode independently samples K_b in {1,...,5}
#   - each episode is right-padded to MAX_CTX_ROWS = 29
#   - padded rows are excluded as keys/values with src_key_padding_mask
#   - prediction is read from each episode's final valid Uc-query row
#
# Composite and non-composite MF baselines are NOT part of this script
# and should remain unchanged.
# ============================================================

import os
import glob
import time
import json
import random
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 0. CONFIG
# ============================================================

SEED = 42

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

INPUT_ROOT = (
    "/kaggle/input/notebooks/xinyuanli913/dataset-ice-sheet-final"
)

DATASET_FILENAME = (
    "howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz"
)

NPZ_PATH_FIXED = os.environ.get(
    "NPZ_PATH_FIXED",
    "",
).strip()

RUN_TAG = (
    "ICL_Howard_VERSION_A_t50_20paired_50test_"
    "sharedVelEnc_d512_L5_FF1536_"
    "mixedK_perEpisode_pad29_mask_testK5_final"
)

SAVE_DIR = os.environ.get(
    "SAVE_DIR",
    (
        f"/kaggle/working/{RUN_TAG}"
        if os.path.isdir("/kaggle/working")
        else f"./{RUN_TAG}"
    ),
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True,
)


# ------------------------------------------------------------
# Data-budget protocol: unchanged
# ------------------------------------------------------------

N_PAIRED_TRAIN = 20
N_TEST_EXPECTED = 50

LF_SIDE_EXPECTED = 15
HF_SIDE_EXPECTED = 41
N_COMP = 2


# ------------------------------------------------------------
# ICL context
#
# demo  = [XY, beta, H, Uc, Uf]   -> 5 rows/demo
# query = [XY, beta, H, Uc]       -> 4 rows
#
# L_K = 5 K + 4
# K_max=5 -> MAX_CTX_ROWS=29
# ------------------------------------------------------------

MIN_DEMOS = 1
MAX_DEMOS = 5
TEST_FIXED_DEMOS = 5

ROWS_PER_DEMO = 5
QUERY_ROWS = 4

MAX_CTX_ROWS = (
    ROWS_PER_DEMO
    * MAX_DEMOS
    + QUERY_ROWS
)

assert MAX_CTX_ROWS == 29


# ------------------------------------------------------------
# Training schedule: unchanged
# ------------------------------------------------------------

BATCH_SIZE = 64
EPOCHS = 180
STEPS_PER_EPOCH = 600
PRINT_EVERY = 5

TEST_DIAGNOSTIC_EVERY = 30

N_TRAIN_EVAL_CASES = 256
EVAL_BATCH_SIZE = 128


# ------------------------------------------------------------
# Optimizer / scheduler: unchanged
# ------------------------------------------------------------

WEIGHT_DECAY = 2e-4

BASE_LR = 1.5e-5
MAX_LR = 5.0e-5
MIN_LR = 1.0e-6

WARMUP_FRAC = 0.08
GRAD_CLIP = 1.0


# ------------------------------------------------------------
# Model: unchanged
# ------------------------------------------------------------

D_MODEL = 512
N_HEAD = 8
N_LAYERS = 5
D_FF = 1536
DROPOUT = 0.10

USE_AMP = (
    DEVICE == "cuda"
)

EPS = 1e-8

SAVE_PERIODIC_EVERY = 30


# ============================================================
# 1. ROW TYPES
# ============================================================

TYPE_XY_DEMO = 0
TYPE_BETA_DEMO = 1
TYPE_H_DEMO = 2
TYPE_UC_DEMO = 3
TYPE_UF_DEMO = 4

TYPE_XY_QUERY = 5
TYPE_BETA_QUERY = 6
TYPE_H_QUERY = 7
TYPE_UC_QUERY = 8

N_ROW_TYPES = 9


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

torch.manual_seed(
    SEED
)

np.random.seed(
    SEED
)

random.seed(
    SEED
)

if DEVICE == "cuda":

    torch.cuda.manual_seed_all(
        SEED
    )

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


# Dedicated episode-sampling RNG.
rng_train = np.random.default_rng(
    SEED + 100
)


# ============================================================
# 3. FIND DATASET
# ============================================================

def resolve_dataset_path():

    if NPZ_PATH_FIXED:

        p = Path(
            NPZ_PATH_FIXED
        )

        if not p.exists():

            raise FileNotFoundError(
                f"NPZ_PATH_FIXED does not exist: {p}"
            )

        return str(
            p
        )

    patterns = [
        os.path.join(
            INPUT_ROOT,
            "**",
            DATASET_FILENAME,
        ),

        f"/kaggle/input/**/*{DATASET_FILENAME}",

        f"/mnt/data/{DATASET_FILENAME}",

        f"./{DATASET_FILENAME}",
    ]

    matches = []

    for pat in patterns:

        matches.extend(
            glob.glob(
                pat,
                recursive=True,
            )
        )

    matches = sorted(
        set(matches)
    )

    if not matches:

        raise FileNotFoundError(
            f"Could not find {DATASET_FILENAME}\n"
            f"under INPUT_ROOT={INPUT_ROOT}\n"
            "Set NPZ_PATH_FIXED if needed."
        )

    in_root = [
        m
        for m in matches
        if os.path.abspath(m).startswith(
            os.path.abspath(
                INPUT_ROOT
            )
        )
    ]

    return (
        in_root[0]
        if in_root
        else matches[0]
    )


NPZ_PATH = (
    resolve_dataset_path()
)


print(
    "=" * 122
)

print(
    "VERSION-A HOWARD BENCHMARK | s=0.35 | ICL MIXED-K PER-EPISODE PAD29"
)

print(
    "=" * 122
)

print(
    "Device       :",
    DEVICE,
)

print(
    "Dataset      :",
    NPZ_PATH,
)

print(
    "Save dir     :",
    SAVE_DIR,
)

print(
    "Train K      : independently sampled per episode from 1..5"
)

print(
    "Padded rows  :",
    MAX_CTX_ROWS,
)

print(
    "Padding mask : src_key_padding_mask"
)

print(
    "Test K       :",
    TEST_FIXED_DEMOS,
)

print(
    "Official ckpt: FINAL FIXED-BUDGET CHECKPOINT ONLY"
)

print(
    "=" * 122
)


# ============================================================
# 4. LOAD VERSION-A DATASET
# ============================================================

data = np.load(
    NPZ_PATH,
    allow_pickle=True,
)


required_keys = [
    "x_lf",
    "y_lf",
    "x_hf",
    "y_hf",
    "beta_lf_train",
    "H_lf_train",
    "U_lf_train",
    "beta_hf_train",
    "H_hf_train",
    "U_hf_train",
    "beta_lf_test",
    "H_lf_test",
    "U_lf_test",
    "beta_hf_test",
    "H_hf_test",
    "U_hf_test",
]


for key in required_keys:

    if key not in data:

        raise KeyError(
            f"Missing required Version-A key: {key}"
        )


x_lf = np.asarray(
    data["x_lf"],
    dtype=np.float32,
).reshape(-1)

y_lf = np.asarray(
    data["y_lf"],
    dtype=np.float32,
).reshape(-1)

x_hf = np.asarray(
    data["x_hf"],
    dtype=np.float32,
).reshape(-1)

y_hf = np.asarray(
    data["y_hf"],
    dtype=np.float32,
).reshape(-1)


beta_lf_train_all = np.asarray(
    data["beta_lf_train"],
    dtype=np.float32,
)

H_lf_train_all = np.asarray(
    data["H_lf_train"],
    dtype=np.float32,
)

U_lf_train_all = np.asarray(
    data["U_lf_train"],
    dtype=np.float32,
)


beta_hf_train = np.asarray(
    data["beta_hf_train"],
    dtype=np.float32,
)

H_hf_train = np.asarray(
    data["H_hf_train"],
    dtype=np.float32,
)

U_hf_train = np.asarray(
    data["U_hf_train"],
    dtype=np.float32,
)


beta_lf_test = np.asarray(
    data["beta_lf_test"],
    dtype=np.float32,
)

H_lf_test = np.asarray(
    data["H_lf_test"],
    dtype=np.float32,
)

U_lf_test = np.asarray(
    data["U_lf_test"],
    dtype=np.float32,
)


beta_hf_test = np.asarray(
    data["beta_hf_test"],
    dtype=np.float32,
)

H_hf_test = np.asarray(
    data["H_hf_test"],
    dtype=np.float32,
)

U_hf_test = np.asarray(
    data["U_hf_test"],
    dtype=np.float32,
)


# ------------------------------------------------------------
# Shape audit
# ------------------------------------------------------------

assert x_lf.size == LF_SIDE_EXPECTED
assert y_lf.size == LF_SIDE_EXPECTED

assert x_hf.size == HF_SIDE_EXPECTED
assert y_hf.size == HF_SIDE_EXPECTED


assert beta_lf_train_all.shape[0] >= N_PAIRED_TRAIN
assert H_lf_train_all.shape[0] >= N_PAIRED_TRAIN
assert U_lf_train_all.shape[0] >= N_PAIRED_TRAIN


assert beta_hf_train.shape[0] == N_PAIRED_TRAIN
assert H_hf_train.shape[0] == N_PAIRED_TRAIN
assert U_hf_train.shape[0] == N_PAIRED_TRAIN


assert beta_hf_test.shape[0] == N_TEST_EXPECTED
assert H_hf_test.shape[0] == N_TEST_EXPECTED
assert U_hf_test.shape[0] == N_TEST_EXPECTED

assert U_lf_test.shape[0] == N_TEST_EXPECTED


assert beta_lf_train_all.shape[1:] == (
    LF_SIDE_EXPECTED,
    LF_SIDE_EXPECTED,
)

assert H_lf_train_all.shape[1:] == (
    LF_SIDE_EXPECTED,
    LF_SIDE_EXPECTED,
)

assert U_lf_train_all.shape[1:] == (
    2,
    LF_SIDE_EXPECTED,
    LF_SIDE_EXPECTED,
)


assert beta_hf_train.shape[1:] == (
    HF_SIDE_EXPECTED,
    HF_SIDE_EXPECTED,
)

assert H_hf_train.shape[1:] == (
    HF_SIDE_EXPECTED,
    HF_SIDE_EXPECTED,
)

assert U_hf_train.shape[1:] == (
    2,
    HF_SIDE_EXPECTED,
    HF_SIDE_EXPECTED,
)


assert U_lf_test.shape[1:] == (
    2,
    LF_SIDE_EXPECTED,
    LF_SIDE_EXPECTED,
)

assert U_hf_test.shape[1:] == (
    2,
    HF_SIDE_EXPECTED,
    HF_SIDE_EXPECTED,
)


# ------------------------------------------------------------
# ICL uses ONLY the 20 paired training trajectories.
# ------------------------------------------------------------

beta_lf_train = (
    beta_lf_train_all[
        :N_PAIRED_TRAIN
    ]
)

H_lf_train = (
    H_lf_train_all[
        :N_PAIRED_TRAIN
    ]
)

U_lf_train = (
    U_lf_train_all[
        :N_PAIRED_TRAIN
    ]
)


N_TRAIN = (
    N_PAIRED_TRAIN
)

N_TEST = (
    N_TEST_EXPECTED
)

N_SIDE = (
    HF_SIDE_EXPECTED
)

N_GRID = (
    N_SIDE
    * N_SIDE
)

NX_SCALAR = (
    N_GRID
)

NX_XY = (
    2
    * N_GRID
)

NX_VEL = (
    2
    * N_GRID
)


# ============================================================
# 5. LF 15x15 -> HF 41x41 REPRESENTATION
# ============================================================

def bilinear_upsample_velocity(
    arr_lf,
):

    t = torch.from_numpy(
        np.asarray(
            arr_lf,
            dtype=np.float32,
        )
    )

    out = F.interpolate(
        t,
        size=(
            HF_SIDE_EXPECTED,
            HF_SIDE_EXPECTED,
        ),
        mode="bilinear",
        align_corners=True,
    )

    return (
        out
        .numpy()
        .astype(
            np.float32,
            copy=False,
        )
    )


def bilinear_upsample_scalar(
    arr_lf,
):

    t = torch.from_numpy(
        np.asarray(
            arr_lf,
            dtype=np.float32,
        )
    )[
        :,
        None,
        :,
        :,
    ]

    out = F.interpolate(
        t,
        size=(
            HF_SIDE_EXPECTED,
            HF_SIDE_EXPECTED,
        ),
        mode="bilinear",
        align_corners=True,
    )[
        :,
        0,
    ]

    return (
        out
        .numpy()
        .astype(
            np.float32,
            copy=False,
        )
    )


Uc_train_grid = (
    bilinear_upsample_velocity(
        U_lf_train
    )
)

Uc_test_grid = (
    bilinear_upsample_velocity(
        U_lf_test
    )
)


# As in the previous reference ICL run:
# use HF-grid beta and H as model inputs.
beta_train_grid = (
    beta_hf_train
)

H_train_grid = (
    H_hf_train
)

Uf_train_grid = (
    U_hf_train
)


beta_test_grid = (
    beta_hf_test
)

H_test_grid = (
    H_hf_test
)

Uf_test_grid = (
    U_hf_test
)


# Pairing diagnostic for beta.
beta_lf_train_up = (
    bilinear_upsample_scalar(
        beta_lf_train
    )
)

beta_lf_test_up = (
    bilinear_upsample_scalar(
        beta_lf_test
    )
)


beta_pair_rel_train = (
    np.linalg.norm(
        (
            beta_lf_train_up
            - beta_train_grid
        ).reshape(
            N_TRAIN,
            -1,
        ),
        axis=1,
    )
    /
    np.maximum(
        np.linalg.norm(
            beta_train_grid.reshape(
                N_TRAIN,
                -1,
            ),
            axis=1,
        ),
        1e-12,
    )
)


beta_pair_rel_test = (
    np.linalg.norm(
        (
            beta_lf_test_up
            - beta_test_grid
        ).reshape(
            N_TEST,
            -1,
        ),
        axis=1,
    )
    /
    np.maximum(
        np.linalg.norm(
            beta_test_grid.reshape(
                N_TEST,
                -1,
            ),
            axis=1,
        ),
        1e-12,
    )
)


# HF XY row.
X_hf, Y_hf = np.meshgrid(
    x_hf,
    y_hf,
    indexing="xy",
)

row_xy_raw = np.concatenate(
    [
        X_hf.reshape(-1),
        Y_hf.reshape(-1),
    ],
    axis=0,
).astype(
    np.float32
)


# Flatten physical arrays.
beta_train = (
    beta_train_grid
    .reshape(
        N_TRAIN,
        N_GRID,
    )
    .astype(
        np.float32,
        copy=False,
    )
)

H_train = (
    H_train_grid
    .reshape(
        N_TRAIN,
        N_GRID,
    )
    .astype(
        np.float32,
        copy=False,
    )
)

Uc_train = (
    Uc_train_grid
    .reshape(
        N_TRAIN,
        NX_VEL,
    )
    .astype(
        np.float32,
        copy=False,
    )
)

Uf_train = (
    Uf_train_grid
    .reshape(
        N_TRAIN,
        NX_VEL,
    )
    .astype(
        np.float32,
        copy=False,
    )
)


beta_test = (
    beta_test_grid
    .reshape(
        N_TEST,
        N_GRID,
    )
    .astype(
        np.float32,
        copy=False,
    )
)

H_test = (
    H_test_grid
    .reshape(
        N_TEST,
        N_GRID,
    )
    .astype(
        np.float32,
        copy=False,
    )
)

Uc_test = (
    Uc_test_grid
    .reshape(
        N_TEST,
        NX_VEL,
    )
    .astype(
        np.float32,
        copy=False,
    )
)

Uf_test = (
    Uf_test_grid
    .reshape(
        N_TEST,
        NX_VEL,
    )
    .astype(
        np.float32,
        copy=False,
    )
)


for name, arr in [
    (
        "beta_train",
        beta_train,
    ),
    (
        "H_train",
        H_train,
    ),
    (
        "Uc_train",
        Uc_train,
    ),
    (
        "Uf_train",
        Uf_train,
    ),
    (
        "beta_test",
        beta_test,
    ),
    (
        "H_test",
        H_test,
    ),
    (
        "Uc_test",
        Uc_test,
    ),
    (
        "Uf_test",
        Uf_test,
    ),
]:

    if not np.isfinite(
        arr
    ).all():

        raise ValueError(
            f"Non-finite values found in {name}"
        )


print(
    "\n[data audit]"
)

print(
    "  ICL paired train cases      :",
    N_TRAIN,
)

print(
    "  LF-only train cases ignored :",
    (
        beta_lf_train_all.shape[0]
        - N_TRAIN
    ),
)

print(
    "  Held-out test queries       :",
    N_TEST,
)

print(
    "  LF velocity raw/upscaled    :",
    U_lf_train.shape,
    "->",
    Uc_train_grid.shape,
)

print(
    "  HF velocity                 :",
    Uf_train_grid.shape,
)

print(
    "  beta/H ICL input grid       : HF 41x41"
)

print(
    "  beta LF->HF pairing rel mean train/test: "
    f"{100*beta_pair_rel_train.mean():.6f}% / "
    f"{100*beta_pair_rel_test.mean():.6f}%"
)

print(
    "  IMPORTANT: held-out test Uf is NEVER used as a demonstration."
)


# ============================================================
# 6. TRAIN-ONLY NORMALIZATION
# ============================================================

def scalar_stats_train(
    arr,
):

    vals = arr.astype(
        np.float64,
        copy=False,
    )

    mean = float(
        vals.mean()
    )

    std = float(
        vals.std()
        + 1e-6
    )

    return (
        mean,
        std,
    )


def velocity_component_stats_train(
    arr,
):

    means = []
    stds = []

    for ci in range(2):

        a = (
            ci
            * N_GRID
        )

        b = (
            (ci + 1)
            * N_GRID
        )

        vals = arr[
            :,
            a:b,
        ].astype(
            np.float64,
            copy=False,
        )

        means.append(
            float(
                vals.mean()
            )
        )

        stds.append(
            float(
                vals.std()
                + 1e-6
            )
        )

    return (
        np.asarray(
            means,
            np.float32,
        ),
        np.asarray(
            stds,
            np.float32,
        ),
    )


def normalize_velocity_np(
    arr,
    means,
    stds,
):

    out = np.empty_like(
        arr,
        dtype=np.float32,
    )

    out[
        :,
        :N_GRID,
    ] = (
        arr[
            :,
            :N_GRID,
        ]
        - means[0]
    ) / stds[0]

    out[
        :,
        N_GRID:,
    ] = (
        arr[
            :,
            N_GRID:,
        ]
        - means[1]
    ) / stds[1]

    return out


beta_mean, beta_std = (
    scalar_stats_train(
        beta_train
    )
)

H_mean, H_std = (
    scalar_stats_train(
        H_train
    )
)

Uc_comp_mean, Uc_comp_std = (
    velocity_component_stats_train(
        Uc_train
    )
)

Uf_comp_mean, Uf_comp_std = (
    velocity_component_stats_train(
        Uf_train
    )
)


x_part = (
    row_xy_raw[
        :N_GRID
    ]
)

y_part = (
    row_xy_raw[
        N_GRID:
    ]
)


xy_comp_mean = np.asarray(
    [
        x_part.mean(),
        y_part.mean(),
    ],
    dtype=np.float32,
)

xy_comp_std = np.asarray(
    [
        x_part.std()
        + 1e-6,

        y_part.std()
        + 1e-6,
    ],
    dtype=np.float32,
)


beta_train_n = (
    (
        beta_train
        - beta_mean
    )
    / beta_std
).astype(
    np.float32
)

H_train_n = (
    (
        H_train
        - H_mean
    )
    / H_std
).astype(
    np.float32
)

Uc_train_n = (
    normalize_velocity_np(
        Uc_train,
        Uc_comp_mean,
        Uc_comp_std,
    )
)

Uf_train_n = (
    normalize_velocity_np(
        Uf_train,
        Uf_comp_mean,
        Uf_comp_std,
    )
)


beta_test_n = (
    (
        beta_test
        - beta_mean
    )
    / beta_std
).astype(
    np.float32
)

H_test_n = (
    (
        H_test
        - H_mean
    )
    / H_std
).astype(
    np.float32
)

Uc_test_n = (
    normalize_velocity_np(
        Uc_test,
        Uc_comp_mean,
        Uc_comp_std,
    )
)

Uf_test_n = (
    normalize_velocity_np(
        Uf_test,
        Uf_comp_mean,
        Uf_comp_std,
    )
)


xy_n = np.empty_like(
    row_xy_raw,
    dtype=np.float32,
)

xy_n[
    :N_GRID
] = (
    x_part
    - xy_comp_mean[0]
) / xy_comp_std[0]

xy_n[
    N_GRID:
] = (
    y_part
    - xy_comp_mean[1]
) / xy_comp_std[1]


Uf_comp_mean_t = torch.tensor(
    Uf_comp_mean,
    dtype=torch.float32,
    device=DEVICE,
)

Uf_comp_std_t = torch.tensor(
    Uf_comp_std,
    dtype=torch.float32,
    device=DEVICE,
)


print(
    "\n[normalization: TRAIN 20 only]"
)

print(
    f"  beta mean/std : {beta_mean:.6f} / {beta_std:.6f}"
)

print(
    f"  H    mean/std : {H_mean:.6f} / {H_std:.6f}"
)

print(
    "  Uc means/stds :",
    Uc_comp_mean.tolist(),
    Uc_comp_std.tolist(),
)

print(
    "  Uf means/stds :",
    Uf_comp_mean.tolist(),
    Uf_comp_std.tolist(),
)


# ============================================================
# 7. METRICS / LOSS
# ============================================================

def summarize_array(
    arr,
    scale=1.0,
):

    x = np.asarray(
        arr,
        dtype=np.float64,
    ) * scale

    return {
        "mean":
            float(
                np.mean(x)
            ),

        "median":
            float(
                np.median(x)
            ),

        "min":
            float(
                np.min(x)
            ),

        "max":
            float(
                np.max(x)
            ),

        "std":
            float(
                np.std(x)
            ),

        "p90":
            float(
                np.percentile(
                    x,
                    90.0,
                )
            ),
    }


def rel_l2_np(
    pred,
    target,
    eps=1e-12,
):

    return (
        np.linalg.norm(
            pred
            - target,
            axis=1,
        )
        /
        (
            np.linalg.norm(
                target,
                axis=1,
            )
            + eps
        )
    )


def r2_np(
    pred,
    target,
    eps=1e-12,
):

    y_mean = np.mean(
        target,
        axis=1,
        keepdims=True,
    )

    sse = np.sum(
        (
            pred
            - target
        ) ** 2,
        axis=1,
    )

    sst = (
        np.sum(
            (
                target
                - y_mean
            ) ** 2,
            axis=1,
        )
        + eps
    )

    return (
        1.0
        - sse / sst
    )


def unnormalize_velocity_torch(
    u_n,
):

    u = (
        u_n[
            :,
            :N_GRID,
        ]
        * Uf_comp_std_t[0]
        + Uf_comp_mean_t[0]
    )

    v = (
        u_n[
            :,
            N_GRID:,
        ]
        * Uf_comp_std_t[1]
        + Uf_comp_mean_t[1]
    )

    return torch.cat(
        [
            u,
            v,
        ],
        dim=1,
    )


def balanced_component_mse(
    pred_n,
    target_n,
):

    pred_u = (
        pred_n[
            :,
            :N_GRID,
        ]
    )

    pred_v = (
        pred_n[
            :,
            N_GRID:,
        ]
    )

    tgt_u = (
        target_n[
            :,
            :N_GRID,
        ]
    )

    tgt_v = (
        target_n[
            :,
            N_GRID:,
        ]
    )

    loss_u = torch.mean(
        (
            pred_u
            - tgt_u
        ) ** 2
    )

    loss_v = torch.mean(
        (
            pred_v
            - tgt_v
        ) ** 2
    )

    loss = (
        0.5
        * (
            loss_u
            + loss_v
        )
    )

    return {
        "loss_total":
            loss,

        "loss_mse":
            loss.detach(),

        "loss_u":
            loss_u.detach(),

        "loss_v":
            loss_v.detach(),
    }


# Exact raw LF baseline on the same 50 held-out queries.
raw_uc_rel = (
    rel_l2_np(
        Uc_test,
        Uf_test,
    )
)

raw_uc_mse = np.mean(
    (
        Uc_test
        - Uf_test
    ) ** 2,
    axis=1,
)


print(
    "\n[exact 50-query raw LF baseline]"
)

print(
    "  Upsampled Uc -> Uf RelL2%: "
    f"mean={100*raw_uc_rel.mean():.4f}, "
    f"median={100*np.median(raw_uc_rel):.4f}, "
    f"p90={100*np.percentile(raw_uc_rel,90):.4f}, "
    f"max={100*raw_uc_rel.max():.4f}"
)

print(
    f"  Upsampled Uc -> Uf MSE mean: {raw_uc_mse.mean():.6f}"
)


# ============================================================
# 8. EPISODE / CONTEXT BUILDERS
# ============================================================

def _get_case_arrays(
    split,
    idx,
):

    if split == "train":

        return (
            beta_train_n[idx],
            H_train_n[idx],
            Uc_train_n[idx],
            Uf_train_n[idx],
            Uf_train[idx],
        )

    if split == "test":

        return (
            beta_test_n[idx],
            H_test_n[idx],
            Uc_test_n[idx],
            Uf_test_n[idx],
            Uf_test[idx],
        )

    raise ValueError(
        split
    )


def build_one_context(
    demo_indices,
    q_idx,
    q_split,
):
    """
    Construct ONLY the valid rows for one episode.

    Demos always come from the paired 20-case TRAIN pool.
    The query may come from train (optimization/monitor) or test.
    """

    d_cur = int(
        len(
            demo_indices
        )
    )

    assert (
        MIN_DEMOS
        <= d_cur
        <= MAX_DEMOS
    )

    if q_split == "train":

        assert (
            int(q_idx)
            not in {
                int(x)
                for x
                in demo_indices
            }
        )

    valid_len = (
        ROWS_PER_DEMO
        * d_cur
        + QUERY_ROWS
    )

    mat_ctx = np.zeros(
        (
            valid_len,
            NX_VEL,
        ),
        dtype=np.float32,
    )

    row_types = np.empty(
        valid_len,
        dtype=np.int64,
    )

    k = 0

    for di in demo_indices:

        di = int(
            di
        )

        (
            b_n,
            h_n,
            uc_n_i,
            uf_n_i,
            _,
        ) = _get_case_arrays(
            "train",
            di,
        )

        mat_ctx[
            k,
            :,
        ] = (
            xy_n
        )

        row_types[
            k
        ] = (
            TYPE_XY_DEMO
        )

        k += 1


        mat_ctx[
            k,
            :N_GRID,
        ] = (
            b_n
        )

        row_types[
            k
        ] = (
            TYPE_BETA_DEMO
        )

        k += 1


        mat_ctx[
            k,
            :N_GRID,
        ] = (
            h_n
        )

        row_types[
            k
        ] = (
            TYPE_H_DEMO
        )

        k += 1


        mat_ctx[
            k,
            :,
        ] = (
            uc_n_i
        )

        row_types[
            k
        ] = (
            TYPE_UC_DEMO
        )

        k += 1


        mat_ctx[
            k,
            :,
        ] = (
            uf_n_i
        )

        row_types[
            k
        ] = (
            TYPE_UF_DEMO
        )

        k += 1


    (
        qb_n,
        qh_n,
        quc_n,
        quf_n,
        quf_phys,
    ) = _get_case_arrays(
        q_split,
        int(q_idx),
    )


    mat_ctx[
        k,
        :,
    ] = (
        xy_n
    )

    row_types[
        k
    ] = (
        TYPE_XY_QUERY
    )

    k += 1


    mat_ctx[
        k,
        :N_GRID,
    ] = (
        qb_n
    )

    row_types[
        k
    ] = (
        TYPE_BETA_QUERY
    )

    k += 1


    mat_ctx[
        k,
        :N_GRID,
    ] = (
        qh_n
    )

    row_types[
        k
    ] = (
        TYPE_H_QUERY
    )

    k += 1


    # Final valid input row = Uc_query.
    query_pos = int(
        k
    )

    mat_ctx[
        k,
        :,
    ] = (
        quc_n
    )

    row_types[
        k
    ] = (
        TYPE_UC_QUERY
    )

    k += 1


    assert (
        k
        == valid_len
    )

    assert (
        query_pos
        == valid_len - 1
    )

    return (
        mat_ctx,
        row_types,
        quf_n.astype(
            np.float32,
            copy=False,
        ),
        quf_phys.astype(
            np.float32,
            copy=False,
        ),
        query_pos,
    )


# ------------------------------------------------------------
# NEW: mixed K independently per episode + right-padding to 29.
# ------------------------------------------------------------

def make_train_batch(
    batch_size,
):

    B = int(
        batch_size
    )

    k_arr = rng_train.integers(
        MIN_DEMOS,
        MAX_DEMOS + 1,
        size=B,
        dtype=np.int64,
    )

    mat_ctx = np.zeros(
        (
            B,
            MAX_CTX_ROWS,
            NX_VEL,
        ),
        dtype=np.float32,
    )

    # Padded positions may carry any valid row-type ID because they are
    # completely masked as keys/values. Zero keeps the array simple.
    row_type_arr = np.zeros(
        (
            B,
            MAX_CTX_ROWS,
        ),
        dtype=np.int64,
    )

    # PyTorch src_key_padding_mask convention:
    # True = padding / ignored as key-value.
    padding_mask = np.ones(
        (
            B,
            MAX_CTX_ROWS,
        ),
        dtype=bool,
    )

    query_pos = np.empty(
        B,
        dtype=np.int64,
    )

    target_n = np.empty(
        (
            B,
            NX_VEL,
        ),
        dtype=np.float32,
    )


    for i in range(B):

        K_i = int(
            k_arr[i]
        )

        idxs = rng_train.choice(
            N_TRAIN,
            size=K_i + 1,
            replace=False,
        )

        demos = (
            idxs[
                :K_i
            ]
        )

        q_idx = int(
            idxs[-1]
        )

        (
            ctx_i,
            type_i,
            tgt_n_i,
            _,
            qpos_i,
        ) = build_one_context(
            demos,
            q_idx,
            "train",
        )

        valid_len = int(
            ctx_i.shape[0]
        )

        assert (
            valid_len
            == ROWS_PER_DEMO
            * K_i
            + QUERY_ROWS
        )

        assert (
            qpos_i
            == valid_len - 1
        )

        mat_ctx[
            i,
            :valid_len,
        ] = (
            ctx_i
        )

        row_type_arr[
            i,
            :valid_len,
        ] = (
            type_i
        )

        padding_mask[
            i,
            :valid_len,
        ] = False

        query_pos[
            i
        ] = (
            qpos_i
        )

        target_n[
            i
        ] = (
            tgt_n_i
        )


    # Strong audit: each episode must have exactly L_K valid rows.
    valid_counts = np.sum(
        ~padding_mask,
        axis=1,
    )

    expected_counts = (
        ROWS_PER_DEMO
        * k_arr
        + QUERY_ROWS
    )

    if not np.array_equal(
        valid_counts,
        expected_counts,
    ):

        raise RuntimeError(
            "Padding-mask valid lengths do not match 5K+4."
        )


    return {
        "k":
            torch.from_numpy(
                k_arr
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "mat_ctx":
            torch.from_numpy(
                mat_ctx
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "row_types":
            torch.from_numpy(
                row_type_arr
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "padding_mask":
            torch.from_numpy(
                padding_mask
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "query_pos":
            torch.from_numpy(
                query_pos
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "target_n":
            torch.from_numpy(
                target_n
            ).to(
                DEVICE,
                non_blocking=True,
            ),
    }


def make_fixed_train_monitor_cases(
    n_cases,
    seed,
    fixed_demos,
):

    r = np.random.default_rng(
        seed
    )

    cases = []

    for _ in range(
        n_cases
    ):

        idxs = r.choice(
            N_TRAIN,
            size=fixed_demos + 1,
            replace=False,
        )

        cases.append(
            (
                idxs[
                    :fixed_demos
                ].astype(
                    np.int64
                ),

                int(
                    idxs[-1]
                ),

                "train",
            )
        )

    return cases


def make_fixed_test_cases(
    seed,
    fixed_demos,
):
    """
    EXACT 50 held-out physical test queries.

    For each query, K demonstrations are sampled only from the paired
    20-case training pool. Choices are created once and reused.
    """

    r = np.random.default_rng(
        seed
    )

    cases = []

    for q_idx in range(
        N_TEST
    ):

        demos = r.choice(
            N_TRAIN,
            size=fixed_demos,
            replace=False,
        ).astype(
            np.int64
        )

        cases.append(
            (
                demos,
                q_idx,
                "test",
            )
        )

    return cases


train_eval_cases = (
    make_fixed_train_monitor_cases(
        N_TRAIN_EVAL_CASES,
        seed=SEED + 2,
        fixed_demos=TEST_FIXED_DEMOS,
    )
)


test_cases = (
    make_fixed_test_cases(
        seed=SEED + 1,
        fixed_demos=TEST_FIXED_DEMOS,
    )
)


print(
    "\n[fixed evaluation protocol]"
)

print(
    "  train-monitor episodes :",
    len(
        train_eval_cases
    ),
    "(diagnostic only)",
)

print(
    "  held-out test queries  :",
    len(
        test_cases
    ),
    "= exact 50 physical test cases",
)

print(
    "  test K                 :",
    TEST_FIXED_DEMOS,
)

print(
    "  first test query demos :",
    test_cases[0][0].tolist(),
)

print(
    "  first test query index :",
    test_cases[0][1],
)

print(
    "  test demonstration pool: TRAIN indices 0..19 ONLY"
)


# ============================================================
# 9. MODEL
# ============================================================

class RowSequenceSemanticEncoderTransformer(
    nn.Module
):

    def __init__(
        self,
        nx_xy,
        nx_scalar,
        nx_vel,
        max_ctx_rows,
        n_row_types,
        d_model,
        nhead,
        num_layers,
        dim_ff,
        dropout,
    ):

        super().__init__()

        self.nx_xy = (
            nx_xy
        )

        self.nx_scalar = (
            nx_scalar
        )

        self.nx_vel = (
            nx_vel
        )

        self.max_ctx_rows = (
            max_ctx_rows
        )

        self.d_model = (
            d_model
        )


        self.xy_encoder = nn.Linear(
            nx_xy,
            d_model,
        )

        self.beta_encoder = nn.Linear(
            nx_scalar,
            d_model,
        )

        self.H_encoder = nn.Linear(
            nx_scalar,
            d_model,
        )

        # Shared Uc/Uf encoder exactly as before.
        self.velocity_encoder = nn.Linear(
            nx_vel,
            d_model,
        )


        self.row_type_emb = nn.Embedding(
            n_row_types,
            d_model,
        )

        self.pos_emb = nn.Embedding(
            max_ctx_rows,
            d_model,
        )


        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )


        self.encoder = nn.TransformerEncoder(
            enc_layer,
            num_layers=num_layers,
        )


        def make_head():

            return nn.Sequential(
                nn.LayerNorm(
                    d_model
                ),

                nn.Linear(
                    d_model,
                    d_model,
                ),

                nn.GELU(),

                nn.Linear(
                    d_model,
                    N_GRID,
                ),
            )


        self.u_head = (
            make_head()
        )

        self.v_head = (
            make_head()
        )


        self.register_buffer(
            "pos_ids",
            torch.arange(
                max_ctx_rows,
                dtype=torch.long,
            ),
            persistent=False,
        )


    def forward(
        self,
        mat_ctx,
        row_type_arr,
        query_pos,
        key_padding_mask=None,
    ):

        B, S, NX_ = (
            mat_ctx.shape
        )

        assert (
            NX_
            == self.nx_vel
        )

        assert (
            S
            <= self.max_ctx_rows
        )


        flat_ctx = mat_ctx.reshape(
            B * S,
            NX_,
        )

        flat_types = row_type_arr.reshape(
            B * S
        )


        xy_mask = (
            (
                flat_types
                == TYPE_XY_DEMO
            )
            |
            (
                flat_types
                == TYPE_XY_QUERY
            )
        )


        beta_mask = (
            (
                flat_types
                == TYPE_BETA_DEMO
            )
            |
            (
                flat_types
                == TYPE_BETA_QUERY
            )
        )


        H_mask = (
            (
                flat_types
                == TYPE_H_DEMO
            )
            |
            (
                flat_types
                == TYPE_H_QUERY
            )
        )


        vel_mask = (
            (
                flat_types
                == TYPE_UC_DEMO
            )
            |
            (
                flat_types
                == TYPE_UF_DEMO
            )
            |
            (
                flat_types
                == TYPE_UC_QUERY
            )
        )


        # Same AMP-safe encoding logic as previous model.
        xy_encoded = self.xy_encoder(
            flat_ctx[
                xy_mask,
                :self.nx_xy,
            ]
        )


        h_flat = xy_encoded.new_zeros(
            (
                B * S,
                self.d_model,
            )
        )


        h_flat[
            xy_mask
        ] = (
            xy_encoded
        )


        if torch.any(
            beta_mask
        ):

            h_flat[
                beta_mask
            ] = self.beta_encoder(
                flat_ctx[
                    beta_mask,
                    :self.nx_scalar,
                ]
            )


        if torch.any(
            H_mask
        ):

            h_flat[
                H_mask
            ] = self.H_encoder(
                flat_ctx[
                    H_mask,
                    :self.nx_scalar,
                ]
            )


        if torch.any(
            vel_mask
        ):

            h_flat[
                vel_mask
            ] = self.velocity_encoder(
                flat_ctx[
                    vel_mask,
                    :self.nx_vel,
                ]
            )


        h = h_flat.reshape(
            B,
            S,
            self.d_model,
        )


        h = (
            h
            + self.row_type_emb(
                row_type_arr
            )
        )


        h = (
            h
            + self.pos_emb(
                self.pos_ids[
                    :S
                ]
            )[
                None,
                :,
                :,
            ]
        )


        # No causal mask.
        # True entries in key_padding_mask are excluded as keys/values.
        h = self.encoder(
            h,
            src_key_padding_mask=
                key_padding_mask,
        )


        batch_idx = torch.arange(
            B,
            device=h.device,
        )


        # Each episode reads from its OWN final valid Uc-query row.
        h_query = h[
            batch_idx,
            query_pos,
            :,
        ]


        pred_u = (
            self.u_head(
                h_query
            )
        )

        pred_v = (
            self.v_head(
                h_query
            )
        )


        return torch.cat(
            [
                pred_u,
                pred_v,
            ],
            dim=1,
        )


model = RowSequenceSemanticEncoderTransformer(
    nx_xy=NX_XY,
    nx_scalar=NX_SCALAR,
    nx_vel=NX_VEL,
    max_ctx_rows=MAX_CTX_ROWS,
    n_row_types=N_ROW_TYPES,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=D_FF,
    dropout=DROPOUT,
).to(
    DEVICE
)


n_params = sum(
    p.numel()
    for p
    in model.parameters()
    if p.requires_grad
)


print(
    "\n[model audit]"
)

print(
    "  model type      : RowSequenceSemanticEncoderTransformer"
)

print(
    "  parameters      :",
    f"{n_params:,}",
)

print(
    "  D_MODEL/heads   :",
    D_MODEL,
    "/",
    N_HEAD,
)

print(
    "  layers / D_FF   :",
    N_LAYERS,
    "/",
    D_FF,
)

print(
    "  shared Uc/Uf enc: YES"
)

print(
    "  causal mask     : NO"
)

print(
    "  padding mask    : YES"
)

print(
    "  max context rows:",
    MAX_CTX_ROWS,
)

print(
    "  prediction row  : each episode's final valid Uc-query row"
)

print(
    "  prediction      : direct full Uf"
)

print(
    "  residual target : NO"
)


# ============================================================
# 10. OPTIMIZER / SCHEDULER
# ============================================================

TOTAL_STEPS = (
    EPOCHS
    * STEPS_PER_EPOCH
)

WARMUP_STEPS = max(
    1,
    int(
        WARMUP_FRAC
        * TOTAL_STEPS
    ),
)


opt = torch.optim.AdamW(
    model.parameters(),
    lr=MAX_LR,
    weight_decay=WEIGHT_DECAY,
)


warmup_scheduler = (
    torch.optim.lr_scheduler.LinearLR(
        opt,
        start_factor=
            BASE_LR / MAX_LR,
        end_factor=1.0,
        total_iters=
            WARMUP_STEPS,
    )
)


cosine_scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=max(
            1,
            TOTAL_STEPS
            - WARMUP_STEPS,
        ),
        eta_min=
            MIN_LR,
    )
)


scheduler = (
    torch.optim.lr_scheduler.SequentialLR(
        opt,
        schedulers=[
            warmup_scheduler,
            cosine_scheduler,
        ],
        milestones=[
            WARMUP_STEPS
        ],
    )
)


if DEVICE == "cuda":

    try:

        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=USE_AMP,
        )

        def amp_context():

            return torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP,
            )

    except Exception:

        scaler = torch.cuda.amp.GradScaler(
            enabled=USE_AMP
        )

        def amp_context():

            return torch.cuda.amp.autocast(
                enabled=USE_AMP
            )

else:

    scaler = None

    def amp_context():

        return nullcontext()


print(
    "\n[optimizer]"
)

print(
    "  AdamW weight decay:",
    WEIGHT_DECAY,
)

print(
    "  total steps       :",
    TOTAL_STEPS,
)

print(
    "  warmup steps      :",
    WARMUP_STEPS,
)

print(
    "  LR base/max/min   :",
    BASE_LR,
    MAX_LR,
    MIN_LR,
)

print(
    "  AMP               :",
    USE_AMP,
)


# ============================================================
# 11. EVALUATION
# ============================================================

def make_eval_batch(
    cases_subset,
):
    """
    Supports arbitrary K values in a subset by using the SAME pad29
    representation. Official reference-benchmark evaluation is K=5.
    """

    B = len(
        cases_subset
    )


    mat_ctx = np.zeros(
        (
            B,
            MAX_CTX_ROWS,
            NX_VEL,
        ),
        dtype=np.float32,
    )


    row_types = np.zeros(
        (
            B,
            MAX_CTX_ROWS,
        ),
        dtype=np.int64,
    )


    padding_mask = np.ones(
        (
            B,
            MAX_CTX_ROWS,
        ),
        dtype=bool,
    )


    query_pos = np.empty(
        B,
        dtype=np.int64,
    )


    target_n = np.empty(
        (
            B,
            NX_VEL,
        ),
        dtype=np.float32,
    )


    target = np.empty(
        (
            B,
            NX_VEL,
        ),
        dtype=np.float32,
    )


    for i, (
        demos,
        q_idx,
        q_split,
    ) in enumerate(
        cases_subset
    ):

        (
            c,
            rt,
            tn,
            tp,
            qpos_i,
        ) = build_one_context(
            demos,
            int(q_idx),
            q_split,
        )

        valid_len = int(
            c.shape[0]
        )

        mat_ctx[
            i,
            :valid_len,
        ] = c

        row_types[
            i,
            :valid_len,
        ] = rt

        padding_mask[
            i,
            :valid_len,
        ] = False

        query_pos[
            i
        ] = qpos_i

        target_n[
            i
        ] = tn

        target[
            i
        ] = tp


    return {
        "mat_ctx":
            torch.from_numpy(
                mat_ctx
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "row_types":
            torch.from_numpy(
                row_types
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "padding_mask":
            torch.from_numpy(
                padding_mask
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "query_pos":
            torch.from_numpy(
                query_pos
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "target_n":
            torch.from_numpy(
                target_n
            ).to(
                DEVICE,
                non_blocking=True,
            ),

        "target":
            torch.from_numpy(
                target
            ).to(
                DEVICE,
                non_blocking=True,
            ),
    }


@torch.no_grad()
def eval_model_cases(
    cases,
    return_predictions=False,
):

    model.eval()


    mse_list = []
    rel_list = []
    r2_list = []

    u_rel_list = []
    v_rel_list = []

    pred_list = []
    target_list = []


    for start in range(
        0,
        len(cases),
        EVAL_BATCH_SIZE,
    ):

        subset = cases[
            start:
            start + EVAL_BATCH_SIZE
        ]


        batch = make_eval_batch(
            subset
        )


        with amp_context():

            pred_n = model(
                batch[
                    "mat_ctx"
                ],
                batch[
                    "row_types"
                ],
                batch[
                    "query_pos"
                ],
                key_padding_mask=
                    batch[
                        "padding_mask"
                    ],
            )


        pred = (
            unnormalize_velocity_torch(
                pred_n.float()
            )
        )


        target = batch[
            "target"
        ]


        err = (
            pred
            - target
        )


        mse = torch.mean(
            err ** 2,
            dim=1,
        )


        rel = (
            torch.linalg.norm(
                err,
                dim=1,
            )
            /
            torch.linalg.norm(
                target,
                dim=1,
            ).clamp_min(
                EPS
            )
        )


        y_mean = torch.mean(
            target,
            dim=1,
            keepdim=True,
        )


        sse = torch.sum(
            err ** 2,
            dim=1,
        )


        sst = torch.sum(
            (
                target
                - y_mean
            ) ** 2,
            dim=1,
        ).clamp_min(
            EPS
        )


        r2 = (
            1.0
            - sse / sst
        )


        pu = pred[
            :,
            :N_GRID,
        ]

        pv = pred[
            :,
            N_GRID:,
        ]

        tu = target[
            :,
            :N_GRID,
        ]

        tv = target[
            :,
            N_GRID:,
        ]


        urel = (
            torch.linalg.norm(
                pu - tu,
                dim=1,
            )
            /
            torch.linalg.norm(
                tu,
                dim=1,
            ).clamp_min(
                EPS
            )
        )


        vrel = (
            torch.linalg.norm(
                pv - tv,
                dim=1,
            )
            /
            torch.linalg.norm(
                tv,
                dim=1,
            ).clamp_min(
                EPS
            )
        )


        mse_list.append(
            mse.cpu().numpy()
        )

        rel_list.append(
            rel.cpu().numpy()
        )

        r2_list.append(
            r2.cpu().numpy()
        )

        u_rel_list.append(
            urel.cpu().numpy()
        )

        v_rel_list.append(
            vrel.cpu().numpy()
        )


        if return_predictions:

            pred_list.append(
                pred.cpu().numpy()
            )

            target_list.append(
                target.cpu().numpy()
            )


    mse_arr = np.concatenate(
        mse_list
    )

    rel_arr = np.concatenate(
        rel_list
    )

    r2_arr = np.concatenate(
        r2_list
    )

    urel_arr = np.concatenate(
        u_rel_list
    )

    vrel_arr = np.concatenate(
        v_rel_list
    )


    out = {
        "MSE_mean":
            float(
                mse_arr.mean()
            ),

        "RelL2_summary_%":
            summarize_array(
                rel_arr,
                100.0,
            ),

        "R2_summary":
            summarize_array(
                r2_arr,
                1.0,
            ),

        "ComponentRelL2_summary_%":
            {
                "u":
                    summarize_array(
                        urel_arr,
                        100.0,
                    ),

                "v":
                    summarize_array(
                        vrel_arr,
                        100.0,
                    ),
            },
    }


    if return_predictions:

        out[
            "predictions"
        ] = np.concatenate(
            pred_list,
            axis=0,
        ).astype(
            np.float32
        )

        out[
            "targets"
        ] = np.concatenate(
            target_list,
            axis=0,
        ).astype(
            np.float32
        )

        out[
            "rel_per_case"
        ] = rel_arr.astype(
            np.float32
        )


    return out


def print_metrics(
    prefix,
    metrics,
):

    r = metrics[
        "RelL2_summary_%"
    ]

    u = metrics[
        "ComponentRelL2_summary_%"
    ][
        "u"
    ]

    v = metrics[
        "ComponentRelL2_summary_%"
    ][
        "v"
    ]

    r2 = metrics[
        "R2_summary"
    ]


    print(
        f"{prefix}: "
        f"RelL2 mean={r['mean']:.4f}% | "
        f"std={r['std']:.4f}% | "
        f"median={r['median']:.4f}% | "
        f"p90={r['p90']:.4f}% | "
        f"max={r['max']:.4f}% | "
        f"MSE={metrics['MSE_mean']:.6f} | "
        f"R2={r2['mean']:.6f} +/- {r2['std']:.6f} | "
        f"u={u['mean']:.4f}% | "
        f"v={v['mean']:.4f}%"
    )


# ============================================================
# 12. CHECKPOINTS / OUTPUTS
# ============================================================

final_ckpt_path = os.path.join(
    SAVE_DIR,
    f"{RUN_TAG}_FINAL_fixedbudget.pt",
)

history_path = os.path.join(
    SAVE_DIR,
    f"{RUN_TAG}_history.json",
)

metrics_path = os.path.join(
    SAVE_DIR,
    f"{RUN_TAG}_final_metrics.json",
)

pred_path = os.path.join(
    SAVE_DIR,
    f"{RUN_TAG}_final_test_predictions.npz",
)


def get_rng_state():

    return {
        "python":
            random.getstate(),

        "numpy_legacy":
            np.random.get_state(),

        "rng_train":
            rng_train.bit_generator.state,

        "torch":
            torch.get_rng_state(),

        "cuda":
            (
                torch.cuda.get_rng_state_all()
                if DEVICE == "cuda"
                else None
            ),
    }


def save_checkpoint(
    path,
    epoch,
    history,
):

    torch.save(
        {
            "model":
                model.state_dict(),

            "optimizer":
                opt.state_dict(),

            "scheduler":
                scheduler.state_dict(),

            "scaler":
                (
                    scaler.state_dict()
                    if scaler is not None
                    else None
                ),

            "epoch":
                int(
                    epoch
                ),

            "global_step":
                int(
                    epoch
                    * STEPS_PER_EPOCH
                ),

            "rng_state":
                get_rng_state(),

            "norm":
                {
                    "beta_mean":
                        beta_mean,

                    "beta_std":
                        beta_std,

                    "H_mean":
                        H_mean,

                    "H_std":
                        H_std,

                    "Uc_comp_mean":
                        Uc_comp_mean,

                    "Uc_comp_std":
                        Uc_comp_std,

                    "Uf_comp_mean":
                        Uf_comp_mean,

                    "Uf_comp_std":
                        Uf_comp_std,

                    "xy_comp_mean":
                        xy_comp_mean,

                    "xy_comp_std":
                        xy_comp_std,
                },

            "config":
                {
                    "RUN_TAG":
                        RUN_TAG,

                    "NPZ_PATH":
                        NPZ_PATH,

                    "protocol":
                        (
                            "Version A t=50; 20 paired train; "
                            "same 50 held-out test"
                        ),

                    "N_PAIRED_TRAIN":
                        N_PAIRED_TRAIN,

                    "N_TEST":
                        N_TEST,

                    "MIN_DEMOS":
                        MIN_DEMOS,

                    "MAX_DEMOS":
                        MAX_DEMOS,

                    "TEST_FIXED_DEMOS":
                        TEST_FIXED_DEMOS,

                    "BATCH_SIZE":
                        BATCH_SIZE,

                    "EPOCHS":
                        EPOCHS,

                    "STEPS_PER_EPOCH":
                        STEPS_PER_EPOCH,

                    "D_MODEL":
                        D_MODEL,

                    "N_HEAD":
                        N_HEAD,

                    "N_LAYERS":
                        N_LAYERS,

                    "D_FF":
                        D_FF,

                    "DROPOUT":
                        DROPOUT,

                    "MAX_CTX_ROWS":
                        MAX_CTX_ROWS,

                    "training_K_sampling":
                        "independent_per_episode_uniform_1_to_5",

                    "training_batching":
                        "mixed_K_right_pad_to_29",

                    "padding_mask":
                        "src_key_padding_mask_bool_true_is_padding",

                    "prediction_row":
                        "each_episode_final_valid_Uc_query_row",

                    "prediction_mode":
                        "direct_full_Uf_no_delta",

                    "test_checkpoint_selection":
                        "NONE; final fixed-budget only",

                    "test_demo_pool":
                        "20 paired train cases only",

                    "lf_velocity_upsampling":
                        (
                            "bilinear 15x15->41x41 "
                            "align_corners=True"
                        ),

                    "beta_H_source":
                        (
                            "HF-grid beta/H "
                            "for train and test"
                        ),
                },

            "history":
                history,
        },
        path,
    )


# ============================================================
# 13. BATCHING SMOKE TEST
# ============================================================

smoke = make_train_batch(
    BATCH_SIZE
)


assert smoke[
    "mat_ctx"
].shape == (
    BATCH_SIZE,
    MAX_CTX_ROWS,
    NX_VEL,
)

assert smoke[
    "row_types"
].shape == (
    BATCH_SIZE,
    MAX_CTX_ROWS,
)

assert smoke[
    "padding_mask"
].shape == (
    BATCH_SIZE,
    MAX_CTX_ROWS,
)

assert smoke[
    "query_pos"
].shape == (
    BATCH_SIZE,
)


smoke_k = (
    smoke[
        "k"
    ]
    .detach()
    .cpu()
    .numpy()
)

smoke_valid = (
    (
        ~smoke[
            "padding_mask"
        ]
    )
    .sum(
        dim=1
    )
    .detach()
    .cpu()
    .numpy()
)


assert np.array_equal(
    smoke_valid,
    (
        ROWS_PER_DEMO
        * smoke_k
        + QUERY_ROWS
    ),
)


print(
    "\n[batching smoke test]"
)

print(
    "  mat_ctx shape :",
    tuple(
        smoke[
            "mat_ctx"
        ].shape
    ),
)

print(
    "  K values      :",
    sorted(
        set(
            smoke_k.tolist()
        )
    ),
)

print(
    "  valid lengths :",
    sorted(
        set(
            smoke_valid.tolist()
        )
    ),
)

print(
    "  query-pos rule:",
    "query_pos = 5*K + 3",
)

assert np.array_equal(
    (
        smoke[
            "query_pos"
        ]
        .detach()
        .cpu()
        .numpy()
    ),
    (
        ROWS_PER_DEMO
        * smoke_k
        + 3
    ),
)

print(
    "  smoke test    : PASS"
)


# ============================================================
# 14. TRAIN
# ============================================================

print(
    "\n"
    + "=" * 122
)

print(
    "TRAINING START | MIXED-K PER EPISODE | PAD29"
)

print(
    "=" * 122
)

print(
    "Paired ICL train cases        :",
    N_TRAIN,
)

print(
    "Held-out test queries         :",
    N_TEST,
)

print(
    "Training K                    : independently sampled 1..5 per episode"
)

print(
    "Training padded length        :",
    MAX_CTX_ROWS,
)

print(
    "Padding mask                  : YES"
)

print(
    "Testing K                     :",
    TEST_FIXED_DEMOS,
)

print(
    "Demo rows                     : [XY,beta,H,Uc,Uf]"
)

print(
    "Query rows                    : [XY,beta,H,Uc]"
)

print(
    "Target                        : full Uf_q"
)

print(
    "Official model selection      : FINAL STEP ONLY"
)

print(
    "Held-out diagnostics in train : logging only; NEVER selection"
)

print(
    "Total scheduled steps         :",
    TOTAL_STEPS,
)

print(
    "=" * 122
)


history = []

start_time = time.time()


# Optional epoch-0 training-side monitor.
initial_train = eval_model_cases(
    train_eval_cases
)

print_metrics(
    "EPOCH 0 TRAIN MONITOR",
    initial_train,
)


global_k_counts = {
    k: 0
    for k
    in range(
        MIN_DEMOS,
        MAX_DEMOS + 1,
    )
}


for ep in range(
    1,
    EPOCHS + 1,
):

    model.train()

    epoch_t0 = (
        time.time()
    )


    logs = {
        "loss_total": [],
        "loss_mse": [],
        "loss_u": [],
        "loss_v": [],
    }


    epoch_k_counts = {
        k: 0
        for k
        in range(
            MIN_DEMOS,
            MAX_DEMOS + 1,
        )
    }


    for _ in range(
        STEPS_PER_EPOCH
    ):

        batch = make_train_batch(
            BATCH_SIZE
        )


        k_cpu = (
            batch[
                "k"
            ]
            .detach()
            .cpu()
            .numpy()
        )


        for k in range(
            MIN_DEMOS,
            MAX_DEMOS + 1,
        ):

            n_k = int(
                np.sum(
                    k_cpu
                    == k
                )
            )

            epoch_k_counts[
                k
            ] += n_k

            global_k_counts[
                k
            ] += n_k


        opt.zero_grad(
            set_to_none=True
        )


        with amp_context():

            pred_n = model(
                batch[
                    "mat_ctx"
                ],
                batch[
                    "row_types"
                ],
                batch[
                    "query_pos"
                ],
                key_padding_mask=
                    batch[
                        "padding_mask"
                    ],
            )


            loss_dict = (
                balanced_component_mse(
                    pred_n,
                    batch[
                        "target_n"
                    ],
                )
            )


            loss_total = (
                loss_dict[
                    "loss_total"
                ]
            )


        if USE_AMP:

            scaler.scale(
                loss_total
            ).backward()

            scaler.unscale_(
                opt
            )

            nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            scaler.step(
                opt
            )

            scaler.update()

        else:

            loss_total.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            opt.step()


        scheduler.step()


        for key in logs:

            logs[
                key
            ].append(
                float(
                    loss_dict[
                        key
                    ]
                    .detach()
                    .cpu()
                )
            )


    epoch_logs = {
        k:
            float(
                np.mean(v)
            )
        for k, v
        in logs.items()
    }


    should_print = (
        ep == 1
        or ep % PRINT_EVERY == 0
        or ep == EPOCHS
    )


    if should_print:

        train_metrics = (
            eval_model_cases(
                train_eval_cases
            )
        )


        lr = float(
            opt.param_groups[
                0
            ][
                "lr"
            ]
        )


        elapsed_epoch = (
            time.time()
            - epoch_t0
        )


        row = {
            "epoch":
                int(
                    ep
                ),

            "global_step":
                int(
                    ep
                    * STEPS_PER_EPOCH
                ),

            "lr":
                lr,

            "seconds":
                float(
                    elapsed_epoch
                ),

            "elapsed_total_seconds":
                float(
                    time.time()
                    - start_time
                ),

            "epoch_K_counts":
                {
                    str(k):
                        int(v)
                    for k, v
                    in epoch_k_counts.items()
                },

            **{
                f"train_{k}":
                    v
                for k, v
                in epoch_logs.items()
            },

            "train_monitor":
                train_metrics,
        }


        print(
            f"epoch {ep:03d}/{EPOCHS} | "
            f"step {ep*STEPS_PER_EPOCH:06d}/{TOTAL_STEPS} | "
            f"loss={epoch_logs['loss_total']:.6e} | "
            f"trainRel={train_metrics['RelL2_summary_%']['mean']:.4f}% | "
            f"lr={lr:.3e} | "
            f"{elapsed_epoch:.1f}s"
        )


        print(
            "  K counts:",
            epoch_k_counts,
        )


        # Diagnostic only. Never used for selection.
        if (
            ep % TEST_DIAGNOSTIC_EVERY == 0
            or ep == EPOCHS
        ):

            test_diag = (
                eval_model_cases(
                    test_cases
                )
            )

            row[
                "test_diagnostic_only"
            ] = (
                test_diag
            )

            print_metrics(
                "  HELD-OUT TEST DIAGNOSTIC ONLY",
                test_diag,
            )


        history.append(
            row
        )


        with open(
            history_path,
            "w",
        ) as f:

            json.dump(
                history,
                f,
                indent=2,
            )


    if (
        SAVE_PERIODIC_EVERY
        and ep
        % SAVE_PERIODIC_EVERY
        == 0
        and ep
        != EPOCHS
    ):

        periodic_path = os.path.join(
            SAVE_DIR,
            f"{RUN_TAG}_epoch{ep:03d}_resume.pt",
        )

        save_checkpoint(
            periodic_path,
            ep,
            history,
        )

        print(
            "  [resume checkpoint]",
            periodic_path,
        )


# ============================================================
# 15. OFFICIAL FINAL EVALUATION
# ============================================================

save_checkpoint(
    final_ckpt_path,
    EPOCHS,
    history,
)


final_train_metrics = (
    eval_model_cases(
        train_eval_cases
    )
)


final_test = (
    eval_model_cases(
        test_cases,
        return_predictions=True,
    )
)


pred = final_test.pop(
    "predictions"
)

target = final_test.pop(
    "targets"
)

rel_per_case = final_test.pop(
    "rel_per_case"
)


np.savez_compressed(
    pred_path,

    pred=
        pred.astype(
            np.float32
        ),

    target=
        target.astype(
            np.float32
        ),

    rel_l2=
        rel_per_case.astype(
            np.float32
        ),

    test_query_indices=
        np.arange(
            N_TEST,
            dtype=np.int64,
        ),

    test_demo_indices=
        np.stack(
            [
                x[0]
                for x
                in test_cases
            ],
            axis=0,
        ).astype(
            np.int64
        ),
)


final_report = {
    "run_tag":
        RUN_TAG,

    "dataset":
        NPZ_PATH,

    "protocol":
        {
            "version":
                "A_t50",

            "paired_train_cases":
                20,

            "lf_only_train_cases_used":
                0,

            "heldout_test_cases":
                50,

            "train_K":
                "independent per episode, uniform 1..5",

            "training_batching":
                "mixed-K right-pad to 29",

            "padding_mask":
                True,

            "prediction_row":
                "final valid Uc-query row per episode",

            "test_K":
                5,

            "test_demonstrations":
                (
                    "only from paired 20-case "
                    "training pool"
                ),

            "official_checkpoint":
                "final fixed-budget checkpoint",

            "test_used_for_checkpoint_selection":
                False,

            "prediction":
                "direct full Uf",

            "residual_delta_training":
                False,
        },

    "model_parameters":
        int(
            n_params
        ),

    "global_training_K_counts":
        {
            str(k):
                int(v)
            for k, v
            in global_k_counts.items()
        },

    "raw_Uc_test":
        {
            "RelL2_summary_%":
                summarize_array(
                    raw_uc_rel,
                    100.0,
                ),

            "MSE_mean":
                float(
                    raw_uc_mse.mean()
                ),
        },

    "final_train_monitor":
        final_train_metrics,

    "final_test":
        final_test,
}


with open(
    metrics_path,
    "w",
) as f:

    json.dump(
        final_report,
        f,
        indent=2,
    )


print(
    "\n"
    + "=" * 122
)

print(
    "OFFICIAL FINAL VERSION-A ICL RESULT | MIXED-K PAD29 | FINAL FIXED-BUDGET CHECKPOINT"
)

print(
    "=" * 122
)


print_metrics(
    "FINAL TEST",
    final_test,
)


print(
    "RAW Uc TEST: "
    f"RelL2 mean={100*raw_uc_rel.mean():.4f}% | "
    f"MSE={raw_uc_mse.mean():.6f}"
)


print(
    "\nGlobal training K counts:"
)

print(
    global_k_counts
)


print(
    "\nSaved:"
)

print(
    "  final checkpoint :",
    final_ckpt_path,
)

print(
    "  history          :",
    history_path,
)

print(
    "  final metrics    :",
    metrics_path,
)

print(
    "  predictions      :",
    pred_path,
)

print(
    "=" * 122
)


VERSION-A HOWARD BENCHMARK | s=0.35 | ICL MIXED-K PER-EPISODE PAD29
Device       : cuda
Dataset      : /kaggle/input/notebooks/xli46913/canonical-dataset/LAYER1_CANONICAL_gpScale0p35/howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz
Save dir     : /kaggle/working/ICL_Howard_VERSION_A_t50_20paired_50test_sharedVelEnc_d512_L5_FF1536_mixedK_perEpisode_pad29_mask_testK5_final
Train K      : independently sampled per episode from 1..5
Padded rows  : 29
Padding mask : src_key_padding_mask
Test K       : 5
Official ckpt: FINAL FIXED-BUDGET CHECKPOINT ONLY

[data audit]
  ICL paired train cases      : 20
  LF-only train cases ignored : 80
  Held-out test queries       : 50
  LF velocity raw/upscaled    : (20, 2, 15, 15) -> (20, 2, 41, 41)
  HF velocity                 : (20, 2, 41, 41)
  beta/H ICL input grid       : HF 41x41
  beta LF->HF pairing rel mean train/test: 0.960676% / 0.958872%
  IMPORTANT: held-out test Uf is NEVER used as a demonstration.

[normalization: TRAIN 20 only]


/tmp/ipykernel_24/593104249.py:2284: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(



[optimizer]
  AdamW weight decay: 0.0002
  total steps       : 108000
  warmup steps      : 8640
  LR base/max/min   : 1.5e-05 5e-05 1e-06
  AMP               : True

[batching smoke test]
  mat_ctx shape : (64, 29, 3362)
  K values      : [1, 2, 3, 4, 5]
  valid lengths : [9, 14, 19, 24, 29]
  query-pos rule: query_pos = 5*K + 3
  smoke test    : PASS

TRAINING START | MIXED-K PER EPISODE | PAD29
Paired ICL train cases        : 20
Held-out test queries         : 50
Training K                    : independently sampled 1..5 per episode
Training padded length        : 29
Padding mask                  : YES
Testing K                     : 5
Demo rows                     : [XY,beta,H,Uc,Uf]
Query rows                    : [XY,beta,H,Uc]
Target                        : full Uf_q
Official model selection      : FINAL STEP ONLY
Held-out diagnostics in train : logging only; NEVER selection
Total scheduled steps         : 108000
EPOCH 0 TRAIN MONITOR: RelL2 mean=102.6071% | std=0.7765% | medi